In [6]:
#r "nuget: ScottPlot, 5.0.56"
#r "nuget: SkiaSharp.NativeAssets.Linux.NoDependencies, 2.88.9"
#r "../task17/bin/Debug/net10.0/task17.dll"

using System;
using System.IO;
using System.Linq;
using System.Threading;
using ScottPlot;
using task17;


Installed Packages ScottPlot, 5.0.56 SkiaSharp.NativeAssets.Linux.NoDependencies, 2.88.9

In [7]:
public class TimeMarkerCommand : ICommand
{
    public long Timestamp { get; private set; } = -1;
    public void Execute() => Timestamp = DateTime.UtcNow.Ticks;
}

public class MultiStepTask : ICommand
{
    private readonly int _steps;
    private int _completed;
    
    public MultiStepTask(int stepCount) => _steps = stepCount;
    public bool Finished => _completed >= _steps;
    
    public void Execute() 
    { 
        _completed++; 
        Thread.Sleep(1); 
    }
}

public static class ExperimentHelpers
{
    public static double CalculateMedian(double[] data)
    {
        var sorted = data.OrderBy(x => x).ToArray();
        int middle = sorted.Length / 2;
        return sorted.Length % 2 == 0 ? (sorted[middle - 1] + sorted[middle]) / 2.0 : sorted[middle];
    }

    public static double MeasureResponseTime(int concurrentLongTasks)
    {
        var scheduler = new RoundRobinScheduler();
        var server = new ServerThread(scheduler);
        
        long startTime = DateTime.UtcNow.Ticks;
        
        for (int i = 0; i < concurrentLongTasks; i++)
            server.Enqueue(new MultiStepTask(20));
        
        var marker = new TimeMarkerCommand();
        server.Enqueue(marker);
        server.Enqueue(new SoftStopCommand(server));
        
        server.Start();
        server.UnderlyingThread.Join();
        
        return TimeSpan.FromTicks(marker.Timestamp - startTime).TotalMilliseconds;
    }
}

In [12]:
int testRuns = 10;
int[] taskCounts = { 0, 2, 4, 8, 16, 32 };
double[] medianLatencies = new double[taskCounts.Length];


for (int i = 0; i < taskCounts.Length; i++)
{
    double[] measurements = new double[testRuns];
    
    for (int j = 0; j < testRuns; j++)
    {
        measurements[j] = ExperimentHelpers.MeasureResponseTime(taskCounts[i]);
    }
    
    medianLatencies[i] = ExperimentHelpers.CalculateMedian(measurements);
}

File.WriteAllLines("results.txt", 
    new[] { "Результаты измерения планировщика:" }
    .Concat(taskCounts.Select((n, i) => $"задач: {n}, задержка: {medianLatencies[i]} мс"))
);

var plot = new Plot();
var scatterPlot = plot.Add.Scatter(
    taskCounts.Select(x => (double)x).ToArray(), 
    medianLatencies
);

scatterPlot.MarkerSize = 8;
scatterPlot.LineWidth = 2;
scatterPlot.Color = ScottPlot.Colors.Blue; 

plot.XLabel("Number of long tasks");
plot.YLabel("Response time (ms)");
plot.Title("Scheduler efficiency ");
plot.Grid.IsVisible = true;
plot.Axes.Bottom.TickLabelStyle.FontName = "DejaVu Sans";
plot.Axes.Left.TickLabelStyle.FontName = "DejaVu Sans";
plot.Axes.Title.Label.FontName = "DejaVu Sans";
plot.Axes.Bottom.Label.FontName = "DejaVu Sans";
plot.Axes.Left.Label.FontName = "DejaVu Sans";
plot.SavePng("plot.png", 800, 600);
